In [ ]:
%load_ext autoreload
%autoreload 2

from tabulate import tabulate
import torch.nn as nn

from fart.constants import MAGNITUDE
from fart.model.evaluate_model import evaluate_model
from fart.model.train_model import train_model
from fart.model.prepare_datasets import prepare_datasets
from fart.utils import get_data_filepath, get_project_root
from fart.visualization.evaluation_line_chart import evaluation_line_chart
from fart.visualization.learning_curve_chart import learning_curve_chart
from fart.visualization.plot_styles import apply_plot_styles

apply_plot_styles()

In [ ]:
assets_dir = get_project_root() / "assets"
market = "BTC-EUR"
interval = "1d"
data_filepath = get_data_filepath(assets_dir, market, interval)

batch_size = 16
learning_rate = 0.001
num_blocks = 4
num_epochs = 100
num_lags = 100
num_neurons = 20
num_splits = 5
train_size = 0.8

In [ ]:
x_train, y_train, x_test, y_test = prepare_datasets(
    data_filepath=data_filepath,
    target=MAGNITUDE,
    num_lags=num_lags,
    train_size=train_size,
)

In [ ]:
def build_model_fn():
    layers: list[nn.Module] = []
    prev_dim = num_lags

    for _ in range(num_blocks):
        layers.append(nn.Linear(prev_dim, num_neurons))
        layers.append(nn.BatchNorm1d(num_neurons))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(p=0.2))
        prev_dim = num_neurons

    layers.append(nn.Linear(prev_dim, 1))  # Output layer

    return nn.Sequential(*layers)


print(build_model_fn())

In [ ]:
model, results = train_model(
    build_model_fn=build_model_fn,
    x_train=x_train,
    y_train=y_train,
    batch_size=batch_size,
    learning_rate=learning_rate,
    num_epochs=num_epochs,
    num_splits=num_splits,
)

In [ ]:
learning_curve_chart(results)

In [ ]:
(
    y_train_pred,
    y_test_pred,
    accuracy_train,
    accuracy_test,
    rmse_train,
    rmse_test,
    mae_train,
    mae_test,
) = evaluate_model(
    model=model,
    x_train=x_train,
    y_train=y_train,
    x_test=x_test,
    y_test=y_test,
)

print(
    tabulate(
        [
            [
                "Train",
                round(accuracy_train, 2),
                rmse_train,
                mae_train,
            ],
            [
                "Test",
                round(accuracy_test, 2),
                rmse_test,
                mae_test,
            ],
        ],
        headers=["Dataset", "Accuracy", "RMSE", "MAE"],
    )
)

In [ ]:
evaluation_line_chart(
    y_train=y_train,
    y_test=y_test,
    y_pred=y_test_pred,
)

In [ ]:
evaluation_line_chart(
    y_test=y_test,
    y_pred=y_test_pred,
)